## Reading Stream

In [0]:
%sql
CREATE OR REPLACE TABLE motocycles
AS SELECT *
FROM read_files(
    '/Volumes/demoworkspace/default/motostore_dataset/motocycle-json',
    format => 'json',
    multiLine => true
)

In [0]:
%sql
SELECT * FROM motocycles

In [0]:
(spark.readStream
    .table("motocycles")
    .createOrReplaceTempView("motocycles_streaming_tmp_vw")
)

In [0]:
%sql
SELECT * FROM motocycles_streaming_tmp_vw

In [0]:
%sql
SELECT brand, count(motocycle_id) AS total_motocycles
FROM motocycles_streaming_tmp_vw
GROUP BY brand

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW brand_counts_tmp_vw AS (
  SELECT brand, count(motocycle_id) AS total_motocycles
  FROM motocycles_streaming_tmp_vw
  GROUP BY brand
);

## Streaming in Batch Mode

On overide la table avec le mode complete

In [0]:
(spark.table("brand_counts_tmp_vw")                               
      .writeStream  
      .trigger(availableNow=True)
      .outputMode("complete")
      .option("checkpointLocation", "/Volumes/demoworkspace/default/motostore_dataset/motocycle_counts_checkpoint")
      .table("motocycle_counts")
)

In [0]:
%sql
SELECT * FROM motocycle_counts;
    

In [0]:
%sql
DROP TABLE IF EXISTS motocycle_counts;

## Auto Loader

In [0]:
(spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/Volumes/demoworkspace/default/motostore_dataset/orders_schema")
        .option("multiLine", "true")
        .load("/Volumes/demoworkspace/default/motostore_dataset/order-raw")
      .writeStream
        .option("checkpointLocation", "/Volumes/demoworkspace/default/motostore_dataset/orders_checkpoint")
        .table("orders_updates")
)

In [0]:
%sql
SELECT * FROM orders_updates

## Loading new file

In [0]:
%sql
SELECT * FROM orders_updates

In [0]:
%sql
DESCRIBE HISTORY orders_updates

In [0]:
%sql
DROP TABLE orders_updates

In [0]:
dbutils.fs.rm("/Volumes/demoworkspace/default/motostore_dataset/orders_checkpoint", True)